# Passive Shear-Wave Velocity from ATOM-1C Nodes

**Near-Surface Geophysics — Missouri S&T**

You have a set of Geometrics ATOM-1C nodes that sat in the ground recording
nothing in particular for an hour or two. This notebook turns that into a
shear-wave velocity profile, or — just as often, and just as usefully — tells
you that your data cannot support one and explains why.

The steps are:

1. **Read** the `.atm` files and merge the hourly folders into whole deployments.
2. **Check the records** — which nodes worked, when, and how loud.
3. **Check the array** — what wavelengths and depths this geometry can resolve.
4. **Measure dispersion** — phase velocity as a function of frequency.
5. **Invert** for a layered $V_s$ profile.

Units are **metres and seconds** throughout. Times are **UTC**, which is what
the nodes record.

> **The one thing to take away.** Steps 3 and 4 contain tests that can fail.
> A dispersion curve is easy to produce and hard to produce *correctly*: the
> failure modes do not look like noise, they look like tidy curves that happen
> to describe your array rather than the ground. Do not skip the diagnostics
> to get to the colourful plot.

## 0. Setup

Run this cell first. It installs the `shallowgeo` package and the forward
modelling code used by the inversion.

In [ ]:
try:
    import shallowgeo  # noqa: F401
except ImportError:
    %pip install -q "shallow-geophysics[masw] @ git+https://github.com/Maurer-GEMLab/shallow-geophysics.git"

%pip install -q ipywidgets

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.figsize": (11, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

from shallowgeo.drivers.atom_atm import (
    scan_atm, group_deployments, read_atm_deployment, read_atm_file,
)
from shallowgeo import passive as P
from shallowgeo.surfacewave import (
    DispersionCurve, forward_dispersion, initial_model, invert_dispersion,
)
print("ready")

## 1. Load your data

Three ways to get the files in. **Run the one that applies to you.**

The nodes write one file per minute, named `<unit-id><minute>.atm`, into
folders named `YYMMDDHH` in UTC. A single afternoon of recording therefore
spans several folders — that is a storage detail, not a survey boundary, and
step 2 stitches them back together.

In [ ]:
# --- Option A: upload a .zip (Colab) ------------------------------------
DATA_DIR = None
try:
    from google.colab import files
    import zipfile, io, tempfile
    print("Choose your .zip of ATOM-1C data...")
    uploaded = files.upload()
    name = next(iter(uploaded))
    DATA_DIR = Path(tempfile.mkdtemp()) / "atom"
    with zipfile.ZipFile(io.BytesIO(uploaded[name])) as z:
        z.extractall(DATA_DIR)
    print(f"extracted to {DATA_DIR}")
except ImportError:
    print("Not running in Colab - use Option B or C below.")

In [ ]:
# --- Option B: Google Drive (Colab) -------------------------------------
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = Path('/content/drive/MyDrive/ATOM-1C_PassiveShearData')

# --- Option C: a local folder -------------------------------------------
# DATA_DIR = Path('~/data/ATOM-1C_PassiveShearData').expanduser()

In [ ]:
files_found = sorted(p for p in Path(DATA_DIR).rglob("*.atm")
                     if not p.name.startswith("._"))
assert files_found, f"no .atm files under {DATA_DIR}"
print(f"{len(files_found)} .atm files")
print("example:", files_found[0].relative_to(DATA_DIR))

## 2. Inventory and deployment grouping

`scan_atm` reads only the headers, so it is fast enough to run over the whole
dataset before deciding what to load. `group_deployments` then splits the
files into separate field days by looking for gaps in time — folder names are
ignored, because a deployment that crosses midnight or an hour boundary is
still one deployment.

In [ ]:
scan = scan_atm(files_found)
deployments = group_deployments(scan)
display(deployments[["deployment", "date", "n_files", "n_nodes", "nodes",
                     "start", "end", "duration_min", "folders"]])

In [ ]:
# Pick one. Change this number to analyse a different field day.
DEPLOYMENT = int(deployments["deployment"].iloc[-1])

chosen = deployments[deployments.deployment == DEPLOYMENT].iloc[0]
print(f"Deployment {DEPLOYMENT}: {chosen.date}, {chosen.n_nodes} nodes, "
      f"{chosen.duration_min:.0f} minutes, folders {chosen.folders}")

sel = scan[(scan.start_time >= chosen.start) & (scan.start_time <= chosen.end)]
survey = read_atm_deployment(sel.path.tolist())
print(survey)
print(f"{survey.n_traces} nodes x {survey.n_samples} samples at "
      f"{survey.sample_rate:g} Hz, starting {survey.start_time}")

## 3. Are the records any good?

Before anything else: did every node actually record, for the whole time, at a
sensible amplitude?

Three things to look for.

**Coverage** below 1.0 means the node stopped early or started late. A node at
0.2 recorded for a fifth of the deployment, and keeping it will shrink the
window that *all* nodes share down to its own.

**Saturation.** The ADC is 24-bit, so the largest representable sample is
$2^{23} = 8388608$. A node reporting that value was clipping — usually while
somebody was still planting it — and those minutes are not ground motion.

**Background level.** Nodes routinely differ by a factor of 10 or more in RMS
because of coupling and local noise. That is normal and is handled later by
normalising each trace. What is *not* normal is one node being loud in a way
that no other node sees.

In [ ]:
coverage = P.node_coverage(survey)
display(coverage)

FULL_SCALE = 2 ** 23
clipped = coverage[coverage.abs_max >= 0.99 * FULL_SCALE]
if len(clipped):
    print("SATURATED nodes (clipping the 24-bit ADC):",
          ", ".join(clipped.node), "- inspect the RMS timeline below")

In [ ]:
# Per-minute RMS for every node: the single most informative QC plot.
minute = sel.copy()
minute["minute"] = minute.start_time.dt.floor("min")
timeline = minute.pivot_table(index="minute", columns="node", values="rms")

fig, ax = plt.subplots(figsize=(12, 4.5))
for node in timeline.columns:
    ax.semilogy(timeline.index, timeline[node], lw=1.2, label=node)
ax.set_xlabel("UTC"); ax.set_ylabel("RMS (counts)")
ax.set_title(f"Per-minute RMS - deployment {DEPLOYMENT} ({chosen.date})")
ax.legend(ncol=5, fontsize=9)
fig.autofmt_xdate(); plt.show()

print("Spikes that appear on EVERY node at the same minute are real events\n"
      "passing through the array - those are the signal in step 6b.\n"
      "Spikes on one node alone are that node being disturbed.")

### Choose the nodes and the time window

Drop nodes that failed, then trim to the interval where the survivors all
recorded simultaneously. Simultaneity is not a nicety: every transform below
compares phase between nodes, and a gap filled with zeros is a very loud
impulse at both its edges.

In [ ]:
# Keep nodes with good coverage. Edit this list by hand if you disagree.
MIN_COVERAGE = 0.8
NODES = coverage.loc[coverage.coverage >= MIN_COVERAGE, "node"].tolist()
print("keeping:", NODES)
print("dropping:", [n for n in coverage.node if n not in NODES])

t0, t1 = P.common_window(survey, nodes=NODES)
work = P.trim(survey, t0, t1, nodes=NODES)
print(f"\ncommon window {t0:%H:%M:%S} to {t1:%H:%M:%S} UTC "
      f"= {(t1 - t0).total_seconds() / 60:.1f} minutes")
print(f"data matrix {work.data.shape}, missing samples "
      f"{np.isnan(work.data).mean() * 100:.2f}%")

## 4. What can this array resolve?

This is the step that is easiest to skip and most expensive to skip.

An array of receivers can only measure surface waves over a limited band of
wavelengths, set entirely by its geometry:

- **Shortest wavelength**, $\lambda_{\min} = 2 \Delta x_{\min}$. Below two
  stations per wavelength the array cannot tell a short wave from a long one —
  spatial aliasing, the same phenomenon as a wagon wheel turning backwards in
  a film. A transform will still draw energy there, at a wavelength pinned to
  the station spacing.
- **Longest wavelength**, a small multiple of the aperture. Over an array much
  shorter than one wavelength the phase barely changes from one end to the
  other, and the apparent velocity runs off to infinity.

A Rayleigh wave samples to roughly a third of its wavelength, which converts
that band into a depth range. Everything outside it is off-limits no matter
how clean the recording is.

In [ ]:
layout = P.array_layout(work)
limits = P.resolution_limits(layout)
print(layout)
print(f"linearity {layout.linearity:.3f} "
      f"({'line array' if layout.is_linear else '2-D array'}), "
      f"azimuth {layout.azimuth:.0f} deg")
print(limits)

display(layout.separations[["node_a", "node_b", "distance"]].round(1))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.5))
a1.plot(layout.east, layout.north, "o-", ms=9)
for n, e, no in zip(layout.nodes, layout.east, layout.north):
    a1.annotate(n, (e, no), textcoords="offset points", xytext=(6, 6), fontsize=8)
a1.set_aspect("equal"); a1.set_xlabel("east (m)"); a1.set_ylabel("north (m)")
a1.set_title("Array geometry (node GPS, median fix)")

f_axis = np.linspace(1, 40, 200)
for c_ref, style in [(150, ":"), (250, "-"), (400, "--")]:
    a2.plot(f_axis, c_ref / f_axis, style, label=f"c = {c_ref} m/s")
a2.axhspan(limits.lambda_min, limits.lambda_max, color="tab:green", alpha=0.15,
           label="resolvable")
a2.set_yscale("log"); a2.set_xlabel("frequency (Hz)"); a2.set_ylabel("wavelength (m)")
a2.set_title("Resolvable band"); a2.legend(fontsize=8)
plt.tight_layout(); plt.show()

lo, hi = limits.frequency_band(250)
print(f"\nAt an assumed 250 m/s this array is usable from {lo:.1f} to {hi:.1f} Hz,")
print(f"mapping depths of roughly {limits.depth_min:.0f} to {limits.depth_max:.0f} m.")

## 5. Is there a propagating wavefield to measure?

Passive methods assume the ground beneath the array is carrying surface waves
from distant sources. That assumption is testable, and on quiet ground it
often fails — each node ends up recording mostly its own wind and coupling
noise, which is not shared with anything.

The quantity to look at is the **coherency** between pairs of nodes: how much
of what one node records also appears at another, at each frequency. For a
genuine travelling wave this must **fall off with separation** — that decay is
the Bessel function that spatial autocorrelation (SPAC) inverts for velocity.

A coherency that stays flat as separation grows is not a wave. It is either
noise shared by all the nodes, or a wavefield arriving broadside to the array
so that every station sees the same phase.

In [ ]:
spac = P.spac_coherency(work, window=30.0, fmin=0.5, fmax=60.0)
print(spac)
print(f"({spac.n_windows_rejected} windows rejected by the amplitude gate - "
      "these are transients, handled separately in step 6b)")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.5))
for k in range(len(spac.distance)):
    a1.semilogx(spac.frequency, spac.coherency[k], lw=1.3,
                label=f"{spac.pairs.node_a[k]}-{spac.pairs.node_b[k]}"
                      f" ({spac.distance[k]:.0f} m)")
a1.axhline(0, color="k", lw=0.6)
a1.set_xlabel("frequency (Hz)"); a1.set_ylabel(r"Re[coherency]")
a1.set_title("Coherency per station pair"); a1.legend(fontsize=7)

for f_probe in (2.0, 4.0, 8.0, 16.0):
    row = spac.at(f_probe)
    a2.plot(row.distance, row.coherency, "o-", label=f"{f_probe:g} Hz")
a2.axhline(0, color="k", lw=0.6)
a2.set_xlabel("station separation (m)"); a2.set_ylabel(r"Re[coherency]")
a2.set_title("Decay with separation - the test that matters")
a2.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
sep = P.separation_test(spac)
print("VERDICT:", sep.verdict)
print(f"\nusable frequency bins: {int(sep.usable.sum())} of {sep.usable.size}")
print(f"largest/smallest separation in this array: {sep.geometry_ratio:.1f}x")
print("\nIf the per-pair velocities disagree by about that same factor, the")
print("'dispersion curve' is a picture of the station spacing, not the ground.")

SPAC_OK = bool(sep.usable.mean() > 0.15)
print("\n->", "SPAC is worth trying (step 6a)" if SPAC_OK
      else "SPAC is not applicable here - go to step 6b")

## 6a. Dispersion from ambient noise (SPAC)

Only meaningful if step 5 passed. SPAC (Aki, 1957) averages the coherency
over many time windows and inverts

$$\rho(f, r) = J_0\!\left(\frac{2\pi f r}{c(f)}\right)$$

for the phase velocity $c(f)$, one frequency at a time, one station pair at a
time. The spread between pairs is carried through as the uncertainty — when
the pairs disagree, the model does not fit.

There is a second check built into `spac_dispersion`. A curve can pass the
separation test and still be an artefact, and the giveaway is the
**wavelength**: over a band spanning a factor of 20 in frequency, a real
wavelength moves by at least that much. One that sits still while the
frequency sweeps past it comes from a coherency that is flat in frequency, and
inverting a constant through $J_0$ forces $c \propto f$ — which draws a
smooth, plausible, entirely fictitious rising curve.

In [ ]:
spac_curve = None
if SPAC_OK:
    try:
        spac_curve = P.spac_dispersion(spac, test=sep)
        print(f"{spac_curve.n_points} points, "
              f"{spac_curve.frequency.min():.1f}-{spac_curve.frequency.max():.1f} Hz, "
              f"c = {spac_curve.velocity.min():.0f}-{spac_curve.velocity.max():.0f} m/s")
        print(f"wavelengths {spac_curve.wavelength.min():.0f}-"
              f"{spac_curve.wavelength.max():.0f} m")
        print("\nplausible:", spac_curve.metadata.get("plausible"))
        print(spac_curve.metadata.get("wavelength_verdict", ""))
        if not spac_curve.metadata.get("plausible", True):
            print("\n--> REJECTED. Do not invert this curve. Continue to step 6b.")
            spac_curve = None
    except ValueError as exc:
        print("No SPAC curve could be extracted:\n ", exc)
else:
    print("skipped - step 5 said the SPAC assumptions do not hold here")

## 6b. Dispersion from transient events (passive MASW)

When the ambient field is too weak, the transients usually are not. A vehicle
on a nearby road or somebody walking the line sends a broadband surface-wave
train through the array — an active-source experiment in everything but the
trigger. Park and Miller (2008) call this roadside passive MASW.

Two things differ from a hammer shot and both are handled below. There is no
trigger, so the windows have to be found; and the source can be at either end
of the line, so each window is transformed for both directions and the better
answer kept.

`detect_events` scores a window by the **minimum** over nodes of that node's
amplitude relative to its own background. Taking the minimum is the point: it
demands that every node sees the event, which rejects the far more common case
of one node being disturbed on its own.

In [ ]:
EVENT_WINDOW = 4.0     # seconds
EVENT_THRESHOLD = 3.0  # times each node's own background

events = P.detect_events(work, window=EVENT_WINDOW,
                         threshold=EVENT_THRESHOLD, max_events=80)
print(f"{len(events)} windows where every node rose above "
      f"{EVENT_THRESHOLD}x its background")
for e in events[:8]:
    print("  ", e)

if events:
    ev = events[0]
    n = int(EVENT_WINDOW * work.sample_rate)
    block = np.nan_to_num(work.data[:, ev.index * n:(ev.index + 1) * n])
    t = np.arange(block.shape[1]) / work.sample_rate
    order = np.argsort(layout.along)
    fig, ax = plt.subplots(figsize=(11, 5))
    for rank, i in enumerate(order):
        trace = block[i] / (np.abs(block[i]).max() + 1e-9)
        ax.plot(t, trace * 0.45 + rank, lw=0.8)
        ax.text(-0.02 * t[-1], rank, f"{layout.nodes[i]}\n{layout.along[i]:+.0f} m",
                ha="right", va="center", fontsize=8)
    ax.set_yticks([]); ax.set_xlabel("time (s)")
    ax.set_title(f"Strongest event, {ev.start_time:%H:%M:%S} UTC "
                 "(traces ordered along the array, normalised)")
    plt.tight_layout(); plt.show()
    print("Look for a coherent arrival stepping across the traces. Its slope\n"
          "is the apparent velocity - that is what the transform measures.")

In [ ]:
image = P.event_dispersion_image(work, events, fmin=2, fmax=40,
                                 vmin=60, vmax=800, dv=2)
F, V = np.meshgrid(image.frequency, image.velocity, indexing="ij")

fig, ax = plt.subplots(figsize=(11, 5.5))
pc = ax.pcolormesh(F, V, image.image, shading="auto", cmap="magma")
fig.colorbar(pc, ax=ax, label="normalised energy")

# Shade what the array cannot resolve, so nothing is picked there.
f_line = image.frequency
ax.fill_between(f_line, limits.lambda_min * f_line, image.velocity.max(),
                color="w", alpha=0.35, lw=0)
ax.fill_between(f_line, image.velocity.min(), limits.lambda_max * f_line,
                color="w", alpha=0.35, lw=0)
ax.plot(f_line, limits.lambda_min * f_line, "c--", lw=1.5,
        label=f"aliasing limit ($\\lambda$={limits.lambda_min:.0f} m)")
ax.plot(f_line, limits.lambda_max * f_line, "c:", lw=1.5,
        label=f"aperture limit ($\\lambda$={limits.lambda_max:.0f} m)")
ax.set_ylim(image.velocity.min(), image.velocity.max())
ax.set_xlabel("frequency (Hz)"); ax.set_ylabel("phase velocity (m/s)")
ax.set_title(f"Passive MASW dispersion image, {len(events)} events stacked")
ax.legend(loc="upper left", fontsize=8)
plt.show()

print("Only the UNSHADED wedge is trustworthy. Energy in the shaded regions is\n"
      "the array's own geometry: a ridge that follows a straight line through\n"
      "the origin sits at one fixed wavelength and is an aliasing lobe.")

### Cross-check with two stations

The multichannel image is limited by the station spacing. A single pair of
stations is not — two receivers cannot alias each other — so the two-station
cross-spectral phase reaches shorter wavelengths.

What it buys in bandwidth it pays for in the **cycle ambiguity**: the phase is
unwrapped upwards from the lowest frequency, and one missed cycle down there
displaces everything above it. So this is a cross-check, never a replacement.
**Pairs that disagree mean neither is trustworthy.**

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
two_station = {}
for _, row in layout.separations.iterrows():
    pair = (int(row.index_a), int(row.index_b))
    try:
        curve = P.two_station_dispersion(work, events, pair=pair,
                                         fmin=2, fmax=30, min_coherence=0.3)
    except ValueError as exc:
        print(f"  {row.node_a}-{row.node_b} ({row.distance:.0f} m): {exc}")
        continue
    two_station[(row.node_a, row.node_b)] = curve
    ax.plot(curve.frequency, curve.velocity, ".-", ms=4, lw=1,
            label=f"{row.node_a}-{row.node_b} ({row.distance:.0f} m)")

ax.set_ylim(0, 900); ax.set_xlabel("frequency (Hz)")
ax.set_ylabel("apparent phase velocity (m/s)")
ax.set_title("Two-station phase velocity, one line per pair")
ax.legend(fontsize=8); plt.show()

print("Pairs that agree are measuring the ground. Pairs that fly off to\n"
      "thousands of m/s are seeing no phase difference at all, which means\n"
      "the energy is arriving broadside or is not propagating.")

## 7. Pick the dispersion curve

Drag the sliders to bound where the fundamental mode lies, then the picker
takes the energy maximum inside that corridor at each frequency.

Pick **only inside the unshaded wedge**, and pick **one continuous branch**.
The fundamental mode is the lowest-velocity continuous ridge; higher modes sit
above it and inverting a mixture of the two gives a profile that is wrong in a
way the misfit will not reveal.

In [ ]:
# Widest frequency range over which any plausible soil velocity (120-600 m/s)
# falls inside the array's resolvable wavelength band.
f_lo = float(max(image.frequency.min(), 120.0 / limits.lambda_max))
f_hi = float(min(image.frequency.max(), 600.0 / limits.lambda_min))
print(f"array can resolve 120-600 m/s between {f_lo:.1f} and {f_hi:.1f} Hz")

w = dict(continuous_update=False, style={"description_width": "130px"},
         layout=widgets.Layout(width="520px"))
s_fband = widgets.FloatRangeSlider(value=[f_lo, f_hi], min=image.frequency.min(),
    max=image.frequency.max(), step=0.25, description="frequency (Hz)", **w)
s_vband = widgets.FloatRangeSlider(value=[100.0, 500.0], min=image.velocity.min(),
    max=image.velocity.max(), step=5.0, description="velocity corridor", **w)
s_limits = widgets.Checkbox(value=True, description="enforce array limits")
picked = {}

def _pick(fband, vband, enforce):
    global picked
    fsel = (image.frequency >= fband[0]) & (image.frequency <= fband[1])
    fr, vl, sd = [], [], []
    for k in np.flatnonzero(fsel):
        f_k = image.frequency[k]
        lo, hi = vband
        if enforce:
            lo = max(lo, limits.lambda_min * f_k)
            hi = min(hi, limits.lambda_max * f_k)
        band = (image.velocity >= lo) & (image.velocity <= hi)
        # A corridor squeezed thin by the array limits cannot contain a peak,
        # only an edge.
        if band.sum() < 10:
            continue
        row = image.image[k][band]
        v_band = image.velocity[band]
        j = int(np.argmax(row))
        # A maximum sitting on the edge of the corridor is the corridor, not a
        # mode. Keeping these is how picks end up tracking the aliasing limit
        # and drawing a straight line of constant wavelength.
        if j < 2 or j > len(row) - 3:
            continue
        half = v_band[row >= 0.5 * (row.max() + row.min())]
        fr.append(f_k); vl.append(v_band[j])
        sd.append(float(half.max() - half.min()) / 2 if half.size else np.nan)

    fig, ax = plt.subplots(figsize=(11, 5))
    ax.pcolormesh(F, V, image.image, shading="auto", cmap="magma")
    ax.plot(image.frequency, limits.lambda_min * image.frequency, "c--", lw=1.2)
    ax.plot(image.frequency, limits.lambda_max * image.frequency, "c:", lw=1.2)
    if fr:
        ax.errorbar(fr, vl, yerr=sd, fmt="o-", color="lime", ms=4, lw=1.5,
                    ecolor="lime", elinewidth=0.8, capsize=2, label="picked")
        ax.legend(fontsize=9)
    ax.set_ylim(image.velocity.min(), image.velocity.max())
    ax.set_xlabel("frequency (Hz)"); ax.set_ylabel("phase velocity (m/s)")
    ax.set_title(f"{len(fr)} points picked")
    plt.show()

    if fr:
        picked = dict(frequency=np.array(fr), velocity=np.array(vl),
                      velocity_std=np.array(sd))
        print(f"lambda {np.min(np.array(vl)/np.array(fr)):.0f}"
              f"-{np.max(np.array(vl)/np.array(fr)):.0f} m  ->  depths roughly "
              f"{np.min(np.array(vl)/np.array(fr))/3:.0f}"
              f"-{np.max(np.array(vl)/np.array(fr))/3:.0f} m")

display(widgets.interactive(_pick, fband=s_fband, vband=s_vband, enforce=s_limits))

In [ ]:
assert picked, "Adjust the sliders until points are picked, then re-run this cell."
curve = DispersionCurve(picked["frequency"], picked["velocity"],
                        velocity_std=picked["velocity_std"], mode=0)

try:
    check = P.wavelength_test(curve)
except ValueError as exc:
    raise SystemExit(f"Cannot test this curve: {exc}")

print(check)
print(" ", check.verdict)
print(f"\nwavelength span {check.wavelength_span:.1f}x over a frequency span of "
      f"{check.frequency_span:.1f}x  ->  index {check.index:.2f}")
print("index ~1 = uniform half-space, >1 = normal dispersion, ~0 = artefact")

if not check.plausible:
    print("\n*** This curve does not describe a propagating wave. ***")
    print("Re-pick, or accept that this dataset cannot support an inversion.")

## 8. Invert for the shear-wave velocity profile

The forward problem is the Rayleigh-wave dispersion of a stack of layers
(computed with `disba`); the inverse problem adjusts layer velocities and
thicknesses until the modelled curve matches the picked one.

Two properties of this inversion matter more than the answer it prints.

**It is non-unique.** Many profiles fit the same curve within its uncertainty.
The cell after this one demonstrates that by inverting from several starting
models — the spread between the results is a far better error bar than
anything the optimiser reports.

**It has no resolution outside the picked wavelength band.** The profile is
drawn to $\lambda_{\max}/3$ and means nothing below that. Layer boundaries
returned near the bottom of the model are being set by the starting model, not
by the data.

In [ ]:
N_LAYERS = 4  # including the half-space; 3-5 is sensible for a curve this short

start = initial_model(curve, n_layers=N_LAYERS)
print("starting model:")
print("  Vs        ", np.round(start.vs, 0))
print("  thickness ", np.round(start.thickness, 1))

result = invert_dispersion(curve, start)
print("\ninverted model:")
print("  Vs        ", np.round(result.model.vs, 0))
print("  thickness ", np.round(result.model.thickness, 1))
print(f"\nRMS misfit {np.sqrt(np.mean(result.residuals() ** 2)):.1f} m/s "
      f"on picks spanning {curve.velocity.min():.0f}-{curve.velocity.max():.0f} m/s")

In [ ]:
z_max = curve.wavelength.max() / 3

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 5))
a1.errorbar(curve.frequency, curve.velocity, yerr=curve.velocity_std,
            fmt="o", ms=4, color="k", capsize=2, label="picked")
model_curve = forward_dispersion(result.model, curve.frequency)
a1.plot(model_curve.frequency, model_curve.velocity, "-", lw=2,
        color="tab:red", label="model")
a1.set_xlabel("frequency (Hz)"); a1.set_ylabel("phase velocity (m/s)")
a1.set_title("Dispersion fit"); a1.legend()

depth, vs = result.model.profile(zmax=z_max * 1.4)
a2.step(vs, depth, where="post", lw=2, color="tab:red")
a2.axhspan(z_max, z_max * 1.4, color="k", alpha=0.12)
a2.text(a2.get_xlim()[1], z_max * 1.15, " no data", va="center", fontsize=8)
a2.invert_yaxis(); a2.set_xlabel("Vs (m/s)"); a2.set_ylabel("depth (m)")
a2.set_title(f"Vs profile (resolved to ~{z_max:.0f} m)")
plt.tight_layout(); plt.show()

### How non-unique is it?

Same data, different starting models. Everywhere the profiles agree, the data
are in control; everywhere they fan out, the starting model is. Quote the
spread, not the single line above.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5.5))
solutions = []
rng = np.random.default_rng(0)
for trial in range(8):
    perturbed = start.with_vs(start.vs * rng.uniform(0.7, 1.4, start.vs.size))
    try:
        r = invert_dispersion(curve, perturbed)
    except Exception:
        continue
    d, v = r.model.profile(zmax=z_max * 1.4)
    ax.step(v, d, where="post", lw=1, alpha=0.6, color="tab:blue")
    solutions.append(np.interp(np.linspace(0, z_max, 60), d, v))

if solutions:
    grid = np.linspace(0, z_max, 60)
    band = np.array(solutions)
    ax.fill_betweenx(grid, band.min(0), band.max(0), color="tab:blue", alpha=0.15)
    ax.step(*reversed(result.model.profile(zmax=z_max * 1.4)[::-1]), where="post",
            lw=2.5, color="tab:red", label="preferred")
    spread = 100 * (band.max(0) - band.min(0)) / band.mean(0)
    print(f"spread between solutions: {spread.min():.0f}% to {spread.max():.0f}% "
          "of Vs, worst at depth")
ax.axhspan(z_max, z_max * 1.4, color="k", alpha=0.12)
ax.invert_yaxis(); ax.set_xlabel("Vs (m/s)"); ax.set_ylabel("depth (m)")
ax.set_title("Non-uniqueness"); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

## 9. Write up

Fill this in from the numbers above. A result without its limits is not a
result.

In [ ]:
summary = [
    f"DEPLOYMENT      {chosen.date}, {len(NODES)} nodes used of {chosen.n_nodes}",
    f"WINDOW          {t0:%H:%M} to {t1:%H:%M} UTC "
    f"({(t1 - t0).total_seconds() / 60:.0f} min)",
    "ARRAY           " + ("line" if layout.is_linear else "2-D") + ", "
    f"aperture {layout.aperture:.0f} m, spacing "
    f"{layout.separations.distance.min():.0f}-"
    f"{layout.separations.distance.max():.0f} m",
    f"RESOLVABLE      lambda {limits.lambda_min:.0f}-{limits.lambda_max:.0f} m, "
    f"depth {limits.depth_min:.0f}-{limits.depth_max:.0f} m",
    f"AMBIENT FIELD   {sep.verdict}",
    "METHOD USED     " + ("SPAC" if spac_curve is not None else
                          "passive MASW, %d transient events" % len(events)),
    f"CURVE           {curve.n_points} points, "
    f"{curve.frequency.min():.1f}-{curve.frequency.max():.1f} Hz, "
    f"{curve.velocity.min():.0f}-{curve.velocity.max():.0f} m/s",
    f"                wavelength index {check.index:.2f} "
    "(" + ("plausible" if check.plausible else "ARTEFACT") + ")",
    f"Vs PROFILE      {np.round(result.model.vs, 0)} m/s over "
    f"{np.round(result.model.thickness, 1)} m, resolved to ~{z_max:.0f} m",
    f"MISFIT          {np.sqrt(np.mean(result.residuals() ** 2)):.1f} m/s",
]
print("\n".join(summary))

## Designing an array that works

If the diagnostics above rejected your data, the fix is almost always in the
field layout rather than the processing. With five nodes the constraints
fight each other, and you have to choose which one to satisfy.

**A line is the worst shape for passive work.** It cannot tell where the noise
is coming from, and a wavefield arriving broadside produces near-zero phase
difference across every pair — high coherency, no velocity information. Worse,
that failure looks like a strong, clean measurement.

**Use a circle for SPAC.** One node at the centre and four on a circle of
radius $R$ gives pairs at $R$, $R\sqrt{2}$ and $2R$ with azimuthal coverage in
every direction, which is exactly the geometry SPAC's averaging assumes. Take
$R \approx$ half your target depth.

**Match the spacing to the wavelengths you want, not to the tape you brought.**
Minimum spacing sets the shallow limit ($\lambda_{\min} = 2\Delta x$) and
aperture sets the deep limit. With five nodes you cannot have both a 2 m
spacing and an 80 m aperture, so run the array twice at two scales and join
the curves — the small array covers the shallow band, the large one the deep.

**Record long enough.** SPAC needs hundreds of independent windows; an hour at
a quiet site is a reasonable minimum, and more is better.

**Check coherency in the field**, before pulling the nodes. It is a ten-minute
computation and it is the difference between a repeat visit and a lost dataset.

### References

- Aki, K. (1957). Space and time spectra of stationary stochastic waves.
  *Bull. Earthq. Res. Inst.* **35**, 415–457.
- Okada, H. (2003). *The Microtremor Survey Method*. SEG Geophysical
  Monograph 12.
- Louie, J.N. (2001). Faster, better: shear-wave velocity to 100 meters depth
  from refraction microtremor arrays. *BSSA* **91**, 347–364.
- Park, C.B. & Miller, R.D. (2008). Roadside passive multichannel analysis of
  surface waves (MASW). *J. Environ. Eng. Geophys.* **13**, 1–11.
- Chávez-García, F.J., Rodríguez, M. & Stephenson, W.R. (2005). An alternative
  approach to the SPAC analysis of microtremors. *BSSA* **95**, 277–293.
- Foti, S. et al. (2018). Guidelines for the good practice of surface wave
  analysis. *Bull. Earthquake Eng.* **16**, 2367–2420.
- Cox, B.R. & Teague, D.P. (2016). Layering ratios: a systematic approach to
  the inversion of surface wave data. *Geophys. J. Int.* **207**, 422–438.